# 13 · Machine learning: a regression pipeline done properly
Goal: predict a record's **revenue** from information available beforehand. The ML workflow: **baseline → split by time → pipeline → compare → check for leakage → explain.**

In [ ]:
import sys
sys.path.insert(0, "..")          # lets the notebook import common.py from python_practice/
from common import *              # load_operations, check, OUT, ...
import numpy as np
import pandas as pd

df = load_operations()            # reads the dashboard API if it is running, else data_samples/operations_clean.csv
df["not_completed"] = (df["status"] != "Completed").astype(int)
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 1. Split by time, not at random
Tomorrow's model will be used on days it has never seen. So train on the past and test on the most recent days. A random split lets the model peek at the future.

In [ ]:
df["weekday"] = df["record_date"].dt.day_name()
df["month"] = df["record_date"].dt.month
days = np.sort(df["record_date"].unique())
cutoff = days[int(len(days) * 0.8)]
train, test = df[df["record_date"] < cutoff], df[df["record_date"] >= cutoff]
print(f"train {len(train):,} rows up to {str(train.record_date.max())[:10]}   test {len(test):,} rows from {str(test.record_date.min())[:10]}")

NUM = ["units_processed", "duration_minutes", "month"]
CAT = ["category", "weekday"]
FEATURES = NUM + CAT
y_train, y_test = train["revenue"], test["revenue"]

## 2. Always compare with a baseline
A model is only good if it beats a trivial one. Baseline: predict the training average for everything.

In [ ]:
baseline_mae = mean_absolute_error(y_test, np.full(len(y_test), y_train.mean()))
print(f"baseline MAE: ${baseline_mae:,.2f}")

## 3. A pipeline keeps preprocessing and model together
The scaler and encoder are *fitted on training data only* inside the pipeline, so nothing from the test set leaks in.

In [ ]:
def make(model):
    pre = ColumnTransformer([("num", StandardScaler(), NUM), ("cat", OneHotEncoder(handle_unknown="ignore"), CAT)])
    return Pipeline([("pre", pre), ("model", model)])

models = {"ridge": Ridge(), "random_forest": RandomForestRegressor(n_estimators=100, max_depth=8, random_state=0),
          "gradient_boosting": GradientBoostingRegressor(random_state=0)}
rows = []
for name, m in models.items():
    pipe = make(m).fit(train[FEATURES], y_train)
    pred = pipe.predict(test[FEATURES])
    rows.append({"model": name, "MAE": mean_absolute_error(y_test, pred), "R2": r2_score(y_test, pred)})
results = pd.DataFrame(rows).set_index("model").round(3)
results.loc["baseline (mean)"] = [baseline_mae, 0.0]
results

## 4. Leakage: the too-good-to-be-true check
Add `profit` (which is revenue minus cost) as a feature and the score jumps to nearly perfect. That is not skill: the answer is hidden inside the feature. Ask of every feature: **will I actually know this at prediction time?**

In [ ]:
NUM_L = NUM + ["profit"]
pre = ColumnTransformer([("num", StandardScaler(), NUM_L), ("cat", OneHotEncoder(handle_unknown="ignore"), CAT)])
leaky = Pipeline([("pre", pre), ("model", Ridge())]).fit(train[NUM_L + CAT], y_train)
print("R2 with profit as a feature:", round(r2_score(y_test, leaky.predict(test[NUM_L + CAT])), 4), " <- suspicious")

## Your turn 1: beat the baseline
Train **any** model (try different features or settings) and store its test-set mean absolute error in `my_mae`. It must beat the baseline.

In [ ]:
my_mae = None

In [ ]:
check("my_mae beats the baseline", my_mae is not None and my_mae < baseline_mae, "fit a pipeline on train, predict on test, mean_absolute_error(y_test, pred)")
if my_mae is not None: print(f"yours ${my_mae:,.2f} vs baseline ${baseline_mae:,.2f}")

## Your turn 2: explain the model
`rf_pipe` below is a fitted random forest. Use **permutation importance** (shuffle one feature at a time on the *test* set and measure how much the score drops) to find the most important feature. Store its name in `top_feature`.

Note: units and duration move together, so they share credit. Either may come out on top.

In [ ]:
rf_pipe = make(RandomForestRegressor(n_estimators=100, max_depth=8, random_state=0)).fit(train[FEATURES], y_train)
top_feature = None   # hint: permutation_importance(rf_pipe, test[FEATURES], y_test, n_repeats=5, random_state=0)

In [ ]:
check("top_feature is a volume feature", top_feature in {"units_processed", "duration_minutes"}, "importances_mean.argmax() indexes into FEATURES")